# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [41]:
# install Gurobi package in case not done yet:
%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions
import src.functions as abd # self-made functions by Arty, Ben and Dirk... ;-) => call them by starting with "abd."


Note: you may need to restart the kernel to use updated packages.


### Read parameters

In [42]:
# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"
abd.writeToLogs("cycle planning process started", FOLDER_AND_FILE_LOG, deleteHistory=True) # very first log entry deletes old log entries

In [43]:
# load parameters from CSV into dict
params = abd.readParameters(FOLDER_INPUT / "parameters.csv")
abd.writeToLogs("parameters loaded", FOLDER_AND_FILE_LOG)

### Global variables

In [44]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(params["max_cycle_length"])   # max number of cycle weeks
MIN_REST        = int(params["min_rest"])            # min rest time between shifts in minutes
MAX_CONSEC_DAYS = int(params["max_conseq_working_days"])  # max consecutive working days

# variables for hourly fairness and average weekly work hours
AVG_WEEKLY_HOURS    = float(params["avg_weekly_work_hours"])   # target avg weekly work hours
AVG_REFERENCE_WEEKS = int(params["avg_reference_weeks"])       # weeks window for average
MAX_WEEKLY_HOURS    = AVG_WEEKLY_HOURS * AVG_REFERENCE_WEEKS   # total hours over reference window

#MAX_CYCLE_WEEKS = 365  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
MAX_NB_CYLCEs = 3 # number of cycles

DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}



In [45]:
# basic inputs and parameters
cycles = range(1, MAX_NB_CYLCEs+1)
cycleWeeks  = range(1, MAX_CYCLE_WEEKS+1)
Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday (in line with static variable DICT_WEEKDAYS)

# improvements outstanding:
    # use files for parameter input
        # shift definitions => done
        # available staff => not required
        # user objectives: weighted priorities

creating shift objects:

In [46]:

# read input data for shift definitions
data_shiftSet = abd.readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv")

shift_objects = abd.build_shift_objects(data_shiftSet)
print(type(shift_objects))
#shift_objects = shift_objects.loc[shift_objects.index.repeat(shift_objects["required_staff"])] # multiply by required staff

# distuingish work shift from all shifts: 
#   WorkShifts are all shifts imported from the shift set file
#   other shifts are created hard-coded below (freeDay, reserveShift)
WorkShifts = [s.shift_id for s in shift_objects] #object oriented solution
abd.writeToLogs(f"WorkShifts are defined as {WorkShifts}", FOLDER_AND_FILE_LOG)

# additional hard-coded shifts for stand-bys and free days
freeDayShift = Shift.Shift("[freeDay]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           0, 
                           1, 
                           0, 
                           False, 
                           None 
                           )
shift_objects.append(freeDayShift)

reserveShift = Shift.Shift("[reserveShift]", 
                           "dummy for jump shifts", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           1, 
                           5, 
                           0, 
                           True, 
                           None 
                           )
shift_objects.append(reserveShift)

#log
abd.writeDataToLogs(shift_objects, FOLDER_LOGS / "log_shift_object.csv")

Shifts = [s.shift_id for s in shift_objects]
abd.writeToLogs(f"    Shifts are defined as {Shifts}", FOLDER_AND_FILE_LOG)

print(Shifts)


<class 'list'>
['[00day000week]%_%1', '[00day000week]%_%2', '[00day000week]%_%3', '[00day000week]%_%4', '[00day000week]%_%5', '[00day000week]%_%6', '[00day000week]%_%7', '[night000week]%_%1', '[night000week]%_%2', '[night000week]%_%3', '[night000week]%_%4', '[night000week]%_%5', '[00dayweekend]%_%1', '[00dayweekend]%_%2', '[00dayweekend]%_%3', '[00dayweekend]%_%4', '[00dayweekend]%_%5', '[nightweekend]%_%1', '[nightweekend]%_%2', '[nightweekend]%_%3', '[nightweekend]%_%4', '[nightweekend]%_%5', '[testOnly]%_%1', '[testOnly]%_%2', '[testOnly]%_%3', '[freeDay]', '[reserveShift]']


In [47]:
# Pre-compute all shift pairs that violate MIN_REST if scheduled on consecutive days.
# Excludes dummy shifts (freeDay, spareShift) since they have no real start/end times.
# Result: list of (sh1_id, sh2_id) tuples that cannot appear on consecutive days in a snake.
DUMMY_SHIFTS = {"[freeDay]", "[reserveShift]"}
incompatible_pairs = [
    (sh1.shift_id, sh2.shift_id)
    for sh1 in shift_objects if sh1.shift_id not in DUMMY_SHIFTS
    for sh2 in shift_objects if sh2.shift_id not in DUMMY_SHIFTS
    if Shift.rest_minutes_between(sh1, sh2) < MIN_REST
]

In [48]:
# Pre-compute work hours per shift using shift_duration_hours from Shift.py.
# Used in the weekly work time constraint.
# freeDay and spareShift are excluded — they have no real duration.
shift_hours = {
    s.shift_id: Shift.shift_duration_hours(s)
    for s in shift_objects
    if s.shift_id not in DUMMY_SHIFTS
}

### modelling

$x \to$ x  
$y \to$ active\_week  
$z \to$ active\_cycle  

In [49]:
# modelling

modelCycle = gp.Model("SnakeBuilding_simple")

# --- decision variables
# x[c, s, d, sh] = 1 if in cycle c, cycleWeek s, on weekday d, shift sh is assigned
x = modelCycle.addVars(cycles, cycleWeeks, Weekdays, Shifts,
                       vtype=GRB.BINARY, name="x")

# active_cycle[c] = 1 if cycle c is used at all, 0 otherwise
active_cycle = modelCycle.addVars(cycles, vtype=GRB.BINARY, name="active_cycle")

# active_week[c,s] = 1 if cycle c uses cycleWeek s (i.e., at least one shift in that week is active), 0 otherwise
active_week = modelCycle.addVars(cycles, cycleWeeks, vtype=GRB.BINARY, name="active_week")


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{s,d,w}} >= 1    \forall d \in D, \forall w \in W$$

$x_{s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable,
$s: $ snake number,
$d: $ weekday,
$ws: $ work shift


In [50]:
# basic constraints

# ensure a cycleWeek can only be active when the according cylce is active
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(active_week[c,w] - active_cycle[c] <= 0,
                             name=f"WeekImpliesCycle_c{c}_s{w}")

# enforce active_week >= any assignment in that week
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, w, d, sh] for d in Weekdays for sh in Shifts) <=
            len(Weekdays) * len(Shifts) * active_week[c, w],
            name=f"Link_x_activeWeek_c{c}_s{w}_upper"
        )

# active_cycle must be 1 if any week in that cycle is active
for c in cycles:
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) <= MAX_CYCLE_WEEKS * active_cycle[c],
        name=f"Link_activeWeek_activeCycle_upper_c{c}"
    )
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) >= active_cycle[c],
        name=f"Link_activeWeek_activeCycle_lower_c{c}")

for c in range(1, MAX_NB_CYLCEs): # loop from first to second-last entry
    modelCycle.addConstr(active_cycle[c] - active_cycle[c + 1] >= 0,
                         name=f"CycleOrder_c{c}")

for c in cycles: # loop across all possible cycles
    for w in range(1, MAX_CYCLE_WEEKS): # loop from first to second-last entry
        modelCycle.addConstr(active_week[c, w] - active_week[c, w + 1] >= 0,
                             name=f"WeekOrder_c{c}_s{w}")


In [51]:
for d in Weekdays: #loop over all week days
    for ws in WorkShifts: # loop over all work shift
        shift = next(s for s in shift_objects if s.shift_id == ws) # next is an alternative for 'for s in shift_objects: if s.shift_id == ws: shift = s'
#       if DICT_WEEKDAYS[shift.weekdays[0]] <= d <= DICT_WEEKDAYS[shift.weekdays[-1]]:
        if d in [DICT_WEEKDAYS[w] for w in shift.weekdays]: # correction to also cover shifts that appear for non-consecutive week days (eg. Mon, Wed)
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) >= 1,
                        name=f"Cover_day{d}_{ws}")
        else:
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) <= 0,
                        name=f"Cover_day{d}_{ws}")

#Логика: перед добавлением ограничения проверяем входит ли день d в список weekdays этой смены. Если нет ограничение не добавляется, смена в этот день не требуется.
#Logic: Before adding a restriction, we check whether day d is included in the list of weekdays for this shift. If not, the restriction is not added, as no shift is required on that day.

# 1. each shift has to be covered on each day
#for d in Weekdays:
#    for ws in WorkShifts:
#        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
#                    name=f"Cover_day{d}_{ws}")



#### condition c02

In [52]:
### DOUBLE-CHECK - SUM seems corrupt !!!

# 2. each cycle shall have at most one shift per day
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            modelCycle.addConstr(
                gp.quicksum(x[c, w, d, sh] for sh in Shifts) == active_week[c, w],
                name=f"OneShiftPerDay_c{c}_s{w}_d{d}"
            )


#### condition c03

In [53]:
### DOUBLE-CHECK

# 3) at max 5 consecutive working days (ensure time for resting)
for c in cycles:
    for w in cycleWeeks:
        for start in range(1, 8 - MAX_CONSEC_DAYS + 1):
            modelCycle.addConstr(
                gp.quicksum(x[c, w, d, sh] for d in range(start, start + MAX_CONSEC_DAYS) for sh in WorkShifts)
                <= MAX_CONSEC_DAYS,
                name=f"Max{MAX_CONSEC_DAYS}Work_c{c}_s{w}_start{start}"
            )



#### condition c04  

ensure that cycle weeks are activated in ascending order  

$$y_s - y_{s+1} >= 0$$

#### Condition c05

In [54]:
# c05: minimum rest time between consecutive shifts within a snake week.
# If sh1 on day d and sh2 on day d+1 violate MIN_REST, they cannot both be assigned to the same snake.
# Covers days 1-6 only; wrap-around (day 7 -> day 1) not yet modelled. => DONE NOW
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            if d <= 6:  # for Mon to Sat use pairs of day d and d+1
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w, d+1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )
            elif d == 7: # ensure this condition also for move from Sun to Mon (pair of days: d and 1)
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w, 1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )

#### Condition c06

In [55]:
# c06: total work hours per active snake week must not exceed MAX_WEEKLY_HOURS.
# Ensures no snake week accumulates more than the allowed weekly work time.
# Bound scales with active[s] so inactive snakes are not constrained.
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(
               #shift_hours.get(sh, 0) * x[c, w, d, sh] # replaced by more efficient version without "get()":
                shift_hours[sh] * x[c, w, d, sh]
                for d in Weekdays
               #for sh in Shifts if sh in shift_hours # # replaced by more efficient version:
                for sh in shift_hours.keys()
            ) <= MAX_WEEKLY_HOURS * active_week[c, w],
            name=f"MaxWeeklyHours_c{c}_w{w}"
        )

### objective

In [56]:
# set objective function: minimize number of active snakes
modelCycle.setObjective(gp.quicksum(active_week[c, s] for c in cycles for s in cycleWeeks), GRB.MINIMIZE)
# minimize total number of active cycle weeks (weeks across all cycles)

# improvements outstanding:
    # add various weighted objectives => based on user input

#run optimizer
modelCycle.optimize()



Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 PRO 250 w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de
Optimize a model with 1931760 rows, 208053 columns and 5056714 nonzeros (Min)
Model fingerprint: 0x8f05e82f
Model has 1095 linear objective coefficients
Variable types: 0 continuous, 208053 integer (208053 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 1872533 rows and 101835 columns
Presolve time: 1.32s
Presolved: 59227 rows, 106218 columns, 849181 nonzeros
Variable types: 0 continuous, 106218 integer (106218 binary)
Performing another presolve...
Presolve removed 50369 rows and 7666 columns
Presol

### results

In [57]:
# output (raw version, to be improved for better readability)
    # improvements outstanding:
        # write results in file
        # create a shift overview per staff member

if modelCycle.status == GRB.OPTIMAL:
    print("\nminimum number of cycle weeks:", int(modelCycle.objVal))
    abd.writeToLogs(f"successfully finished cycle plan: found an optimal solution using {int(modelCycle.objVal)} cycle weeks",FOLDER_AND_FILE_LOG)
    for c in cycles:
        if active_cycle[c].X == 1:
            print(f"\ncycle {c}:")
            abd.writeToLogs(f"\ncycle {c}:",FOLDER_AND_FILE_LOG)
            for w in cycleWeeks:
                if active_week[c, w].X == 1:
                    print(f"\ncycle week {w}:")
                    abd.writeToLogs(f"\ncycle week {w}:",FOLDER_AND_FILE_LOG)
                    for d in Weekdays:
                        for sh in Shifts:
                            if x[c, w, d, sh].X == 1:
                                print(f"\t{DICT_WEEKDAYS_RETURN[d]}: {sh.split("%_%", 1)[0]}")
                                abd.writeToLogs(f"\tday {d}: {sh.split("%_%", 1)[0]}",FOLDER_AND_FILE_LOG)


# output to file
if modelCycle.status == GRB.OPTIMAL:
    output_string = ""
    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:
        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;\n")
    for c in cycles:
        if active_cycle[c].X == 1:
            for w in cycleWeeks:
                if active_week[c, w].X == 1:
                    for d in Weekdays:
                        for sh in Shifts:
                            if x[c, w, d, sh].X == 1:
                                output_string = output_string + sh.split("%_%", 1)[0] + ";"
                    with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                        file.write(output_string + "\n")
                    output_string = ""




minimum number of cycle weeks: 20

cycle 1:

cycle week 1:
	Mon: [testOnly]
	Tue: [00day000week]
	Wed: [testOnly]
	Thu: [00day000week]
	Fri: [00day000week]
	Sat: [00dayweekend]
	Sun: [00dayweekend]

cycle week 2:
	Mon: [testOnly]
	Tue: [00day000week]
	Wed: [00day000week]
	Thu: [00day000week]
	Fri: [00day000week]
	Sat: [nightweekend]
	Sun: [freeDay]

cycle week 3:
	Mon: [00day000week]
	Tue: [reserveShift]
	Wed: [00day000week]
	Thu: [night000week]
	Fri: [reserveShift]
	Sat: [nightweekend]
	Sun: [reserveShift]

cycle week 4:
	Mon: [night000week]
	Tue: [freeDay]
	Wed: [night000week]
	Thu: [freeDay]
	Fri: [night000week]
	Sat: [freeDay]
	Sun: [00dayweekend]

cycle week 5:
	Mon: [freeDay]
	Tue: [night000week]
	Wed: [freeDay]
	Thu: [00day000week]
	Fri: [testOnly]
	Sat: [00dayweekend]
	Sun: [nightweekend]

cycle week 6:
	Mon: [night000week]
	Tue: [reserveShift]
	Wed: [testOnly]
	Thu: [00day000week]
	Fri: [00day000week]
	Sat: [00dayweekend]
	Sun: [00dayweekend]

cycle week 7:
	Mon: [00day000wee

In [58]:
print(shift_objects[1])

Shift(shift_id='[00day000week]%_%2', description='day shift week', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri'], start=datetime.time(6, 0), end=datetime.time(16, 45), required_staff=7, shift_class=2, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')


In [59]:
print(shift_hours)

{'[00day000week]%_%1': 10.75, '[00day000week]%_%2': 10.75, '[00day000week]%_%3': 10.75, '[00day000week]%_%4': 10.75, '[00day000week]%_%5': 10.75, '[00day000week]%_%6': 10.75, '[00day000week]%_%7': 10.75, '[night000week]%_%1': 14.75, '[night000week]%_%2': 14.75, '[night000week]%_%3': 14.75, '[night000week]%_%4': 14.75, '[night000week]%_%5': 14.75, '[00dayweekend]%_%1': 10.75, '[00dayweekend]%_%2': 10.75, '[00dayweekend]%_%3': 10.75, '[00dayweekend]%_%4': 10.75, '[00dayweekend]%_%5': 10.75, '[nightweekend]%_%1': 14.75, '[nightweekend]%_%2': 14.75, '[nightweekend]%_%3': 14.75, '[nightweekend]%_%4': 14.75, '[nightweekend]%_%5': 14.75, '[testOnly]%_%1': 10.0, '[testOnly]%_%2': 10.0, '[testOnly]%_%3': 10.0}
